# M6.0b Primer: Inductance, Mutual Inductance and Magnetic Energy

Book pointer: chapter 15, sections 15.1 (inductance), 15.2 (mutual inductance), 15.3 (applications of mutual inductance), 15.4 (energy in magnetic fields), 15.5 (magnetic forces).

Second of four primers for module 6. Primer 00a turned a changing magnetic flux into an emf. This primer asks the next question: when the flux comes from a current, how much emf does a changing current produce, in its own circuit and in a neighbour? The answer is two numbers that depend only on geometry, the self-inductance $L$ and the mutual inductance $M$. Module 5 (primer 00c) already used the inductor as a circuit component with $V = L\,dI/dt$ and a value printed on its label. Here the label is computed from the wire's shape, the energy it stores is located in the field around it, and the same numbers give the force between two coils. Every hand result is reproduced with `emlib.induction` and the Biot-Savart routines of module 4.

## 1. Warm-up

Try on paper first. Answers are in section 7.

1. A coil's current rises at $50\,$A/s and the emf across it is $0.2\,$V. What is $L$? What does the sign of the induced emf say about the current change?
2. Why does the inductance of a solenoid grow as $N^2$ and not as $N$?
3. A small loop sits at the centre of a large loop. Is the flux through the small loop per ampere in the large loop equal to the flux through the large loop per ampere in the small loop? Guess before reading section 3.2.
4. Why does a transformer not work on a steady current?
5. A current flows in an inductor. Where is the stored energy, and what is its density?
6. Why can the coupling coefficient $k = M/\sqrt{L_1L_2}$ never exceed 1?
7. Two coaxial loops carry currents in the same sense. Do they attract or repel, and how is the force related to $M$?


## 2. The ideas

**Flux linkage is proportional to current.** The Biot-Savart law is linear: double the current in a wire and the field it makes doubles everywhere. So does the magnetic flux of that field through any surface, including the surface bounded by the wire itself and the surface bounded by any other wire nearby. A coil of $N$ turns is threaded $N$ times by roughly the same flux, so the quantity that enters Faraday's law is the flux linkage $\Lambda = N\Phi$, the flux summed over all the turns. Linearity means $\Lambda = (\text{a number})\times I$, and that number depends only on the shape and size of the wires and their relative position, not on the current.

**Self-inductance.** The number for a circuit's own flux is its self-inductance, $L = \Lambda/I$, measured in henry ($1\,\mathrm{H} = 1\,\mathrm{Wb/A} = 1\,\mathrm{V\,s/A}$). With Faraday's law from primer 00a, a changing current induces an emf $\mathcal{E} = -L\,dI/dt$ in its own circuit. The minus sign is Lenz's law: the induced emf opposes the change of current that causes it. This is the physical origin of the component law $V = L\,dI/dt$ used in module 5, where $V$ is the voltage drop across the inductor in the direction of the current. A long coil of many turns with a large cross-section has a large $L$; a straight wire has a small one; a perfect zero-thickness wire has an infinite one, because its own field diverges near it, and section 3.1 shows how the wire radius cures that.

**Mutual inductance, and why it is symmetric.** The number for the flux that one circuit sends through another is the mutual inductance. Current $I_1$ in circuit 1 puts flux linkage $\Lambda_{21} = M_{21}I_1$ through circuit 2, and a changing $I_1$ induces $\mathcal{E}_2 = -M_{21}\,dI_1/dt$ there. Swap the roles and you get $M_{12}$. A surprising and useful fact is that $M_{12} = M_{21}$ for every pair of circuits, however different. A tiny loop next to a large one sees the large loop's broad field; the large loop sees the tiny loop's concentrated dipole field; the two fluxes per ampere are exactly equal. The reason is visible in the Neumann formula below, which is symmetric in the two wires.

**The Neumann formula as the general recipe.** Flux through a loop equals the line integral of the vector potential around it (Stokes' theorem, M1 primer 00d), and the vector potential of a wire is a simple sum over its segments (M4 primer 00b). Putting the two together gives $M = \dfrac{\mu_0}{4\pi}\oint\oint\dfrac{d\mathbf{l}_1\cdot d\mathbf{l}_2}{|\mathbf{r}_1 - \mathbf{r}_2|}$, a double sum over pairs of wire segments. It needs no surface, no field and no symmetry, which makes it the computational workhorse: `emlib.induction.neumann_M` evaluates it for any two polylines.

**Transformers and coupling.** When two coils share their flux, a changing current in one drives an emf in the other. If all the flux of each coil threads every turn of the other, the voltages are in the ratio of the turns: this is the ideal transformer that steps $230\,$V mains down to $12\,$V. Real coils leak flux. The fraction that is shared is measured by the coupling coefficient $k = M/\sqrt{L_1L_2}$, which lies between $-1$ and $1$. A transformer on an iron core reaches $k \approx 0.99$; a wireless phone charger with coils a few millimetres apart manages a few tenths. Only a changing flux induces an emf, so a transformer passes changes and blocks a steady current.

**Energy in the field.** Building up a current against the back emf costs work, $\tfrac12LI^2$ for one coil and $\tfrac12L_1I_1^2 + MI_1I_2 + \tfrac12L_2I_2^2$ for two. That energy is stored, not lost: switch off the source and the inductor drives the current on for a while, returning it. For a solenoid the stored energy turns out to be exactly $B^2/2\mu_0$ times the volume of the field, and that density holds everywhere. It is the magnetic twin of the electric energy density $\tfrac12\epsilon_0E^2$ of M2 primer 00c. Because the energy can never be negative, $M^2 \le L_1L_2$, which is why $|k| \le 1$.

**Forces from energy.** Two circuits carrying fixed currents push or pull on each other with force $I_1I_2\,dM/dx$ along any displacement $x$. The derivation is energy bookkeeping (section 3.5), and the result says that coils move so as to increase the flux they share. Parallel currents attract. The same idea gives the outward magnetic pressure $B^2/2\mu_0$ on a solenoid's windings, which is what makes high-field magnets hard to build.

**What notebook 02 does.** Notebook M6.2 computes mutual inductances of loops in arbitrary relative positions with the Neumann double sum and, independently, as a flux, checks both against the exact coaxial formula, and uses the results for coupling coefficients and forces between coils. Every ingredient is derived below.


## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk, ellipe
from emlib import biot_savart as bs
from emlib import induction as ind
from emlib.integrators import rk4_ode

MU0 = bs.MU0
plt.rcParams["figure.figsize"] = (6, 4.5)
np.set_printoptions(precision=4, suppress=True)

### 3.1 Self-inductance

Take a circuit carrying current $I$. Its field $\mathbf{B}$ is proportional to $I$, so the flux of $\mathbf{B}$ through a surface bounded by the circuit is proportional to $I$. If the wire winds $N$ times around the same region, each turn bounds that surface once and the flux counts $N$ times. The flux linkage $\Lambda = N\Phi$ is what Faraday's law sees:
$$\mathcal{E} = -\frac{d\Lambda}{dt}, \qquad \Lambda = LI \quad\Longrightarrow\quad \mathcal{E} = -L\frac{dI}{dt}\quad\text{(rigid circuit)}.$$
$L$ is the self-inductance. It is always positive: a current's own field threads its own loop in the direction given by the right-hand rule, so $\Lambda$ and $I$ have the same sign. The induced emf therefore always opposes the change of current: it fights a rising current and props up a falling one. In a circuit diagram that emf appears as a voltage drop $L\,dI/dt$ across the inductor in the direction of the current, which is the component law of M5 primer 00c.

How to compute $L$ by hand: assume a current $I$, find $\mathbf{B}$ (usually with Ampère's law and a symmetry, M4 primer 00b), integrate it over the surface spanned by one turn, multiply by the number of turns, and divide by $I$. The current always cancels.

**Worked example 1 (a solenoid).** Inside a long solenoid of $N$ turns, length $l$ and cross-section $A = \pi R^2$, the field is uniform, $B = \mu_0nI$ with $n = N/l$. The flux through one turn is $\Phi = \mu_0nIA$, and every one of the $N$ turns links it, so
$$\Lambda = N\mu_0\frac{N}{l}IA, \qquad L = \frac{\mu_0N^2A}{l}.$$
The $N^2$ comes from two separate effects: more turns make a stronger field, and more turns link that field. For $N = 500$, $R = 1\,$cm, $l = 10\,$cm: $\mu_0N^2 = 1.2566\times10^{-6}\cdot2.5\times10^5 = 0.31416$, times $A = 3.1416\times10^{-4}\,\mathrm{m}^2$ gives $9.870\times10^{-5}$, and dividing by $0.1\,$m gives $L = 0.987\,$mH. A current changing at $100\,$A/s induces $0.0987\,$V. The formula assumes the field is uniform all the way to the ends. A finite solenoid loses field near its ends, and the standard correction for a uniform current sheet (Nagaoka's coefficient, a closed form in elliptic integrals that the code evaluates) is a factor 0.920 for this shape, $2R/l = 0.2$, giving $0.908\,$mH.

**Worked example 2 (a toroid).** A toroid of $N$ turns on a core of rectangular cross-section, height $h$, inner radius $a$, outer radius $b$. Ampère's law around a circle of radius $r$ inside the core gives $B = \mu_0NI/2\pi r$, and zero outside. The flux through one turn is
$$\Phi = \int_a^b\frac{\mu_0NI}{2\pi r}h\,dr = \frac{\mu_0NIh}{2\pi}\ln\frac ba, \qquad L = \frac{\mu_0N^2h}{2\pi}\ln\frac ba.$$
No end correction is needed because a toroid has no ends: all of its field stays inside. For $N = 1000$, $h = 1\,$cm, $a = 4\,$cm, $b = 6\,$cm: $L = 2\times10^{-7}\cdot10^6\cdot0.01\cdot\ln1.5 = 0.811\,$mH.

**Worked example 3 (a coaxial cable).** An inner conductor of radius $a$ carries $I$ and the outer sheath of radius $b$ carries it back. Between them $B = \mu_0I/2\pi r$, and outside the sheath the two currents cancel. The flux through a rectangle of length $\ell$ spanning the gap is $\dfrac{\mu_0I\ell}{2\pi}\ln\dfrac ba$, so the inductance per unit length is
$$\frac{L}{\ell} = \frac{\mu_0}{2\pi}\ln\frac ba.$$
For $a = 0.5\,$mm and $b = 2.5\,$mm: $2\times10^{-7}\ln5 = 0.322\,\mu$H/m. If the current is spread through a solid inner wire rather than on its surface, field inside the wire adds $\mu_0/8\pi = 0.05\,\mu$H/m; section 3.4 derives that from the energy. Compare the capacitance per unit length of the same geometry, $2\pi\epsilon_0/\ln(b/a)$ from M2 primer 00c: the product $LC$ per unit length squared is $\mu_0\epsilon_0 = 1/c^2$, a first sign that signals on a cable travel at the speed of light (primer 00c).


**Worked example 4 (a ring of real wire).** A single circular loop of radius $R$ looks like the simplest case and is the hardest. The field of a zero-thickness wire near the wire looks like that of a straight wire, $B \approx \mu_0I/2\pi s$ at distance $s$. The flux through the loop includes $\int ds/s$ down to $s = 0$, which diverges logarithmically. An ideal line current has infinite self-inductance. A real wire of radius $a$ cuts the integral off at $s \approx a$, and the standard result for a thin ring with the current spread uniformly over the wire's cross-section is
$$L = \mu_0R\left(\ln\frac{8R}{a} - \frac74\right), \qquad a \ll R.$$
The logarithm is the trace of the divergence: halving the wire radius adds the same amount, $\mu_0R\ln2$, every time. For $R = 5\,$cm: $a = 0.5\,$mm gives $0.310\,\mu$H, $a = 0.1\,$mm gives $0.411\,\mu$H, $a = 0.01\,$mm gives $0.556\,\mu$H. A computation that represents a wire as a line of segments (all of module 4) can compute mutual inductances between separate wires without trouble, but a self-inductance always needs the wire's thickness as an extra input. That is why `emlib.induction` offers `ring_self_L(R, a)` as a closed form rather than a Neumann sum of a loop with itself.

**Worked example 5 (back to module 5: the RL time constant).** The solenoid of worked example 1 is wound from copper wire of radius $0.25\,$mm. Its length is $500\cdot2\pi\cdot1\,\mathrm{cm} = 31.42\,$m, its resistance $\rho\ell/A = 1.68\times10^{-8}\cdot31.42/(\pi\cdot0.25^2\times10^{-6}) = 2.688\,\Omega$. Connected to a battery, the current rises with time constant $\tau = L/R = 0.987\,\mathrm{mH}/2.688\,\Omega = 0.367\,$ms, the RL circuit of M5 primer 00c, now with both numbers computed from the coil's geometry.

The code reproduces all five examples and plots the logarithmic growth of a ring's inductance as its wire gets thinner.


In [ ]:
N, R, l = 500, 0.01, 0.10
L_sol = ind.solenoid_L(N, R, l)
print(f"worked example 1, solenoid: L = mu0 N^2 pi R^2 / l = {L_sol * 1e3:.4f} mH;  emf at dI/dt = 100 A/s: {L_sol * 100:.4f} V")

def nagaoka(R, l):
    '''Factor by which a finite uniform current sheet's inductance falls short of mu0 N^2 pi R^2 / l.'''
    m = 4 * R**2 / (4 * R**2 + l**2)
    return 4 / (3 * np.pi * np.sqrt(1 - m)) * ((1 - m) / m * (ellipk(m) - ellipe(m)) + ellipe(m) - np.sqrt(m))

print(f"  finite-length factor for 2R/l = {2 * R / l:.1f}: {nagaoka(R, l):.4f}, so L = {L_sol * nagaoka(R, l) * 1e3:.4f} mH")
Nt, h_t, a_t, b_t = 1000, 0.01, 0.04, 0.06
print(f"worked example 2, toroid: L = {MU0 * Nt**2 * h_t / (2 * np.pi) * np.log(b_t / a_t) * 1e3:.4f} mH")
a_c, b_c = 0.5e-3, 2.5e-3
print(f"worked example 3, coaxial cable: L per length = {MU0 / (2 * np.pi) * np.log(b_c / a_c) * 1e6:.4f} uH/m"
      f"  (+ {MU0 / (8 * np.pi) * 1e6:.3f} uH/m inside a solid inner wire)")
for a_w in (0.5e-3, 0.1e-3, 0.01e-3):
    print(f"worked example 4, ring of radius 5 cm, wire radius {a_w * 1e3:.2f} mm: L = {ind.ring_self_L(0.05, a_w) * 1e6:.4f} uH")
length = N * 2 * np.pi * R
R_wire = 1.68e-8 * length / (np.pi * 0.25e-3**2)
print(f"worked example 5, {length:.2f} m of copper wire of radius 0.25 mm: R = {R_wire:.3f} ohm, tau = L/R = {L_sol / R_wire * 1e3:.3f} ms")

a_w = np.logspace(-6, -2.5, 50)
plt.semilogx(a_w * 1e3, ind.ring_self_L(0.05, a_w) * 1e6)
plt.xlabel("wire radius a [mm]"); plt.ylabel("L [uH]")
plt.title("ring of radius 5 cm: L grows without limit as a -> 0"); plt.show()

### 3.2 Mutual inductance

Now two circuits. A current $I_1$ in circuit 1 makes a field $\mathbf{B}_1 \propto I_1$, and some of that field threads circuit 2. The flux linkage of circuit 2 is proportional to $I_1$:
$$\Lambda_{21} = N_2\Phi_{21} = M_{21}I_1, \qquad \mathcal{E}_2 = -M_{21}\frac{dI_1}{dt}.$$
$M_{21}$ is the mutual inductance, again in henry and again set by geometry alone. Unlike $L$ it can have either sign. The sign depends on the direction you choose to count positive current around circuit 2: reverse that choice and the flux through it, measured with the right-hand rule, changes sign. In transformer diagrams the convention is fixed by a dot at one end of each winding.

**The Neumann formula.** Write the flux through circuit 2 with Stokes' theorem (M1 primer 00d) as a line integral of the vector potential, $\Phi_{21} = \int_{S_2}\mathbf{B}_1\cdot d\mathbf{S} = \int_{S_2}(\nabla\times\mathbf{A}_1)\cdot d\mathbf{S} = \oint_{C_2}\mathbf{A}_1\cdot d\mathbf{l}_2$. The vector potential of a current loop is a sum over its elements (M4 primer 00b), $\mathbf{A}_1(\mathbf{r}) = \dfrac{\mu_0I_1}{4\pi}\oint_{C_1}\dfrac{d\mathbf{l}_1}{|\mathbf{r} - \mathbf{r}_1|}$. Substitute and divide by $I_1$:
$$M_{21} = \frac{\mu_0}{4\pi}\oint_{C_1}\oint_{C_2}\frac{d\mathbf{l}_1\cdot d\mathbf{l}_2}{|\mathbf{r}_1 - \mathbf{r}_2|}.$$
Every pair of elements, one on each wire, contributes the dot product of the two elements divided by their distance. Nothing in the formula singles out wire 1 as the source and wire 2 as the receiver: swapping them leaves it unchanged. So
$$M_{12} = M_{21} \equiv M,$$
the reciprocity theorem. It is not obvious from the flux picture, where one side is a broad weak field through a small area and the other is a concentrated field through a large area. On a computer the formula is a double sum over segments, $\sum_{i,j}\Delta\mathbf{l}_i\cdot\Delta\mathbf{l}_j/|\mathbf{r}_i - \mathbf{r}_j|$, valid as long as the wires are separated by more than a segment length. For a wire with itself the distance goes to zero and the sum diverges, which is the self-inductance problem of worked example 4.

**Worked example 6 (a small loop at the centre of a large one).** A loop of radius $a$ sits concentric and coplanar with a loop of radius $b \gg a$. The large loop's field at its centre is $\mu_0I/2b$ (M4 primer 00a) and is nearly uniform over the small loop, so $\Phi = \mu_0I\pi a^2/2b$ and
$$M \approx \frac{\mu_0\pi a^2}{2b}.$$
For $a = 1\,$cm and $b = 20\,$cm: $1.2566\times10^{-6}\cdot\pi\cdot10^{-4}/0.4 = 9.870\times10^{-10}\,$H. Reciprocity says that the small loop's dipole field, which is strong close to the small loop, points the opposite way outside it, and spreads over the whole plane, threads the large loop with exactly the same flux per ampere. Computing that flux directly is a hard integral. The Neumann sum does not care which way round it is done.

**Worked example 7 (two coaxial loops far apart).** Loops of radii $a$ and $b$ on a common axis, a distance $d$ apart with $d \gg a, b$. The axial field of loop 1 at distance $d$ is $\mu_0Ia^2/2(a^2 + d^2)^{3/2} \approx \mu_0Ia^2/2d^3$, nearly uniform over loop 2, so
$$M \approx \frac{\mu_0\pi a^2b^2}{2d^3}.$$
The $1/d^3$ is the dipole fall-off. For $a = b = 5\,$cm at $d = 1\,$m: $1.2337\times10^{-11}\,$H. At $d = 0.3\,$m the approximation is 8 percent high, and at $d = a$ it is four times too large, because loop 2 then sees the near field of loop 1, not its dipole field. The exact result for any separation has the closed form
$$M = \mu_0\sqrt{ab}\left[\left(\frac2k - k\right)K(k) - \frac2kE(k)\right], \qquad k^2 = \frac{4ab}{(a + b)^2 + d^2},$$
with $K$ and $E$ the complete elliptic integrals of the first and second kind (a standard result, implemented as `emlib.induction.coaxial_loops_M`).

**Worked example 8 (a coil wound on a solenoid).** A secondary of $N_2$ turns is wound tightly around the middle of the solenoid of worked example 1. Every secondary turn encloses the solenoid's full flux $\mu_0n_1I_1A$, so $\Lambda_{21} = N_2\mu_0n_1I_1A$ and
$$M = \frac{\mu_0N_1N_2A}{l} = L_1\frac{N_2}{N_1}.$$
For $N_2 = 50$: $M = 0.987\,\mathrm{mH}\times50/500 = 98.7\,\mu$H. By reciprocity the short secondary coil, whose own field is a messy bulge around the middle of the long one, links the 500 solenoid turns with exactly the same $98.7\,\mu$H per ampere.

**The coupling coefficient.** The dimensionless measure of how much flux two coils share is
$$k = \frac{M}{\sqrt{L_1L_2}}, \qquad -1 \le k \le 1.$$
Section 3.4 proves the bound from energy. $|k| = 1$ means every field line of each coil threads every turn of the other: no leakage.

**Animation.** Two coaxial loops. The current in loop 1 swells and shrinks as a slow sine, and its field lines thread loop 2. An induced-current marker on loop 2 always points so as to oppose the change in loop 1's current, and the readout shows the emf $-M\,dI_1/dt$. Then loop 2 slides away along the axis, and the readout and the small plot of $M$ against separation show $M$ falling off, eventually as $1/d^3$.


In [ ]:
from emlib import show
show.animation("m6_waves", "CoupledCoils")

In [ ]:
a, b = 0.01, 0.20
M_n = ind.neumann_M(bs.loop(a, 200), bs.loop(b, 400))
print(f"worked example 6, 1 cm loop at the centre of a 20 cm loop: Neumann {M_n:.4e} H,"
      f" mu0 pi a^2 / 2b = {MU0 * np.pi * a**2 / (2 * b):.4e} H, exact {ind.coaxial_loops_M(a, b, 0.0):.4e} H")
a = b = 0.05
print("\nworked example 7, two 5 cm loops on a common axis:")
print("   d [m]     Neumann      exact     dipole approx   dipole / exact")
for d in (0.005, 0.02, 0.05, 0.1, 0.3, 1.0):
    Mn = ind.neumann_M(bs.loop(a, 300), bs.loop(b, 300, centre=(0, 0, d)))
    Me = ind.coaxial_loops_M(a, b, d)
    Md = MU0 * np.pi * a**2 * b**2 / (2 * d**3)
    print(f"  {d:6.3f}   {Mn:.4e}   {Me:.4e}   {Md:.4e}   {Md / Me:8.4f}")
print(f"\nworked example 8, 50 turns on the solenoid of worked example 1: M = {ind.solenoid_L(500, 0.01, 0.10) * 50 / 500 * 1e6:.1f} uH")

d = np.logspace(-3, 0.5, 80)
plt.loglog(d / a, ind.coaxial_loops_M(a, b, d), label="exact")
plt.loglog(d / a, MU0 * np.pi * a**2 * b**2 / (2 * d**3), "--", label="dipole approximation, 1/d^3")
plt.xlabel("d / a"); plt.ylabel("M [H]"); plt.legend(); plt.title("two equal coaxial loops of radius 5 cm"); plt.show()

Read the table. The Neumann sum with 300 segments per loop matches the exact elliptic-integral value to better than two parts in $10^4$ at every separation, and to a few parts in $10^6$ at $d = 5\,$mm, a tenth of the loop radius. The relative error is largest for distant loops, where the sum adds many nearly cancelling contributions. The dipole formula is useless closer than a few radii and good to 1 percent at 20 radii. On the log-log plot the exact curve bends from a gentle logarithmic dependence at small $d$ (two nearly touching rings behave like two parallel wires) to the $1/d^3$ line.

Reciprocity can be checked where no symmetry helps. The cell takes a $5\,$cm loop in the $xy$ plane and a $3\,$cm loop displaced off axis and tilted, and computes $M$ four ways: the Neumann sum in both orders, and the flux of each loop's Biot-Savart field through the other's flat disk, using `emlib.induction.disk_flux`. The two Neumann orders are the same sum added in a different order. The two flux routes are genuinely different calculations, one integrating a broad field over a small disk, the other a concentrated field over a large one.


In [ ]:
l1 = bs.loop(0.05, 300)
l2 = bs.loop(0.03, 300, centre=(0.02, 0.01, 0.04), normal=(0.3, 0.2, 1.0))
M12, M21 = ind.neumann_M(l1, l2), ind.neumann_M(l2, l1)
m1, d1 = bs.segments(l1)
m2, d2 = bs.segments(l2)
flux_21 = ind.disk_flux(lambda p: bs.wire_B(m1, d1, 1.0, p), (0.02, 0.01, 0.04), (0.3, 0.2, 1.0), 0.03, nr=40, nphi=80)
flux_12 = ind.disk_flux(lambda p: bs.wire_B(m2, d2, 1.0, p), (0, 0, 0), (0, 0, 1), 0.05, nr=40, nphi=80)
print(f"Neumann, loop 1 then loop 2: {M12:.6e} H    loop 2 then loop 1: {M21:.6e} H")
print(f"flux of loop 1's field through loop 2, per amp: {flux_21:.6e} H")
print(f"flux of loop 2's field through loop 1, per amp: {flux_12:.6e} H")

All four agree to one or two parts in $10^4$, the accuracy of the disk quadrature. The flux of loop 2's field through loop 1 is the harder one: loop 1's disk passes close to loop 2's wire, where the field is large and varies fast.

### 3.3 Applications of mutual inductance

**The ideal transformer.** Two coils on a closed iron core. The core guides essentially all the flux, so the same flux $\Phi$ threads every turn of both coils ($k = 1$), and assume no resistance. Each turn of either coil then carries an emf $d\Phi/dt$ in magnitude, so
$$V_1 = N_1\frac{d\Phi}{dt}, \quad V_2 = N_2\frac{d\Phi}{dt} \quad\Longrightarrow\quad \frac{V_2}{V_1} = \frac{N_2}{N_1}.$$
With no losses the power in equals the power out, $V_1I_1 = V_2I_2$, so the currents go the other way: $I_1/I_2 = N_2/N_1$. A step-down transformer delivers a larger current at a lower voltage. In terms of inductances, $L \propto N^2$ for coils sharing a core, so $N_2/N_1 = \sqrt{L_2/L_1}$, and $k = 1$ means $M = \sqrt{L_1L_2}$. Everything rests on $d\Phi/dt$, so a steady current, however large, induces nothing in the secondary.

**Worked example 9 (a mains adaptor).** Step $230\,$V RMS down to $12\,$V to run a $24\,$W load. The turns ratio is $230/12 = 19.17$; with $N_2 = 50$ secondary turns the primary needs $958$. The load draws $I_2 = 24/12 = 2.000\,$A, and the primary supplies $I_1 = 24/230 = 0.1043\,$A.

**A real transformer as two coupled RL circuits.** Give the coils resistances, let $R_2$ include the load, and keep the mutual terms. Kirchhoff's loop law around each circuit, with the emf of each coil written as a voltage drop:
$$V_1(t) = R_1I_1 + L_1\frac{dI_1}{dt} + M\frac{dI_2}{dt}, \qquad 0 = R_2I_2 + L_2\frac{dI_2}{dt} + M\frac{dI_1}{dt}.$$
In matrix form, $\begin{pmatrix}L_1 & M\\ M & L_2\end{pmatrix}\dfrac{d}{dt}\begin{pmatrix}I_1\\I_2\end{pmatrix} = \begin{pmatrix}V_1 - R_1I_1\\ -R_2I_2\end{pmatrix}$. The inductance matrix has determinant $L_1L_2(1 - k^2)$, positive for $|k| < 1$, so it can be inverted and the system handed to `rk4_ode` (M5 primer 00c). The load voltage is $R_2|I_2|$; with the sign convention above, $I_2$ comes out negative when the secondary follows the primary, so the code plots $-R_2I_2$.

**Worked example 10 (what a transformer does with a step).** $L_1 = 10\,$mH, $L_2 = 40\,$mH (a turns ratio of 2), $k = 0.95$ so $M = 0.95\sqrt{4\times10^{-4}} = 19.0\,$mH, $R_1 = 1\,\Omega$, load $R_2 = 100\,\Omega$. Apply $10\,$V to the primary at $t = 0$. Two time scales appear. The fast one is set by the leakage: the part of $L_2$ not shared with the primary, $L_2(1 - k^2)$, in series with the load, gives $L_2(1 - k^2)/R_2 = 39\,\mu$s. Within that time the load voltage jumps to nearly $(M/L_1)\cdot10\,\mathrm{V} = 19\,$V. The slow one is $L_1/R_1 = 10\,$ms: the primary current climbs towards $V/R_1 = 10\,$A, the flux stops changing, and the secondary voltage dies away. A transformer passes the change and blocks the steady state. The second half of the cell drives the same transformer with a $1\,$kHz sine of amplitude $10\,$V, integrates until the start-up transient has gone, fits the amplitude of the load voltage, and compares it with the phasor solution of the same two equations, where $d/dt \to j\omega$ (M5 primer 00c). The ideal-transformer answer would be $2\times10\,\mathrm{V} = 20\,$V; leakage and the primary's resistance cost about 11 percent.


In [ ]:
V1_rms, V2_rms, P_load = 230.0, 12.0, 24.0
print(f"worked example 9: turns ratio N1/N2 = {V1_rms / V2_rms:.2f}; with N2 = 50, N1 = {round(50 * V1_rms / V2_rms)};"
      f"  I2 = {P_load / V2_rms:.3f} A, I1 = {P_load / V1_rms:.4f} A")

L1, L2, k, R1, R2 = 10e-3, 40e-3, 0.95, 1.0, 100.0
M = k * np.sqrt(L1 * L2)
Linv = np.linalg.inv(np.array([[L1, M], [M, L2]]))

def coupled(V1):
    return lambda t, I: Linv @ np.array([V1(t) - R1 * I[0], -R2 * I[1]])

t = np.linspace(0, 40e-3, 8001)
I = rk4_ode(coupled(lambda t: 10.0), np.zeros(2), t)
V_load = -R2 * I[:, 1]
print(f"\nworked example 10, 10 V step: M = {M * 1e3:.2f} mH, peak load voltage {V_load.max():.3f} V"
      f" (M/L1 x 10 V = {M / L1 * 10:.3f} V); after 40 ms: {V_load[-1]:.4f} V, primary current {I[-1, 0]:.4f} A")
print(f"  leakage time constant L2 (1 - k^2) / R2 = {L2 * (1 - k**2) / R2 * 1e6:.1f} us, slow time constant L1 / R1 = {L1 / R1 * 1e3:.1f} ms")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(t * 1e3, V_load); ax[0].set_xlabel("t [ms]"); ax[0].set_ylabel("load voltage [V]")
ax[0].set_title("a step on the primary: the secondary sees only the change")
ax[1].plot(t * 1e3, I[:, 0]); ax[1].set_xlabel("t [ms]"); ax[1].set_ylabel("I1 [A]"); ax[1].set_title("the primary current rises to V / R1")
plt.tight_layout(); plt.show()

w = 2 * np.pi * 1000.0
t = np.linspace(0, 80e-3, 16001)
I = rk4_ode(coupled(lambda t: 10.0 * np.sin(w * t)), np.zeros(2), t)
last = t > 70e-3
basis = np.column_stack([np.sin(w * t[last]), np.cos(w * t[last])])
coef, *_ = np.linalg.lstsq(basis, -R2 * I[last, 1], rcond=None)
Z = np.array([[R1 + 1j * w * L1, 1j * w * M], [1j * w * M, R2 + 1j * w * L2]])
I_phasor = np.linalg.solve(Z, np.array([10.0, 0.0]))
print(f"\n1 kHz drive of amplitude 10 V: load voltage amplitude from the ODE {np.hypot(*coef):.4f} V,"
      f" from phasors {abs(R2 * I_phasor[1]):.4f} V; ideal transformer N2/N1 x 10 V = {np.sqrt(L2 / L1) * 10:.1f} V")

The step response peaks at $17.96\,$V rather than $19\,$V because the leakage and the primary's resistance already take a share during the fast rise, and after $40\,$ms (four slow time constants) only $0.39\,$V is left while the primary current has reached $9.80\,$A. Under sinusoidal drive the ODE and the phasor solution agree to the fourth decimal, $17.841\,$V against the ideal $20\,$V.

**Wireless power and the coupling coefficient.** A phone charger is a transformer with no core and an air gap: a flat coil in the pad and another in the phone. Only part of the pad's flux reaches the phone's coil, so $k$ is well below 1 and falls fast with distance. For two equal flat coils of $N$ closely wound turns both $M$ and each $L$ scale roughly as $N^2$, so $k$ is nearly the same as for single rings, which the closed forms give at once: $M$ from `coaxial_loops_M` and $L$ from `ring_self_L`. Practical systems tune both coils to resonance with capacitors (the series RLC of M5 primer 00c), and the best possible efficiency then depends on $k^2Q_1Q_2$, with $Q$ the quality factors. Low $k$ can be compensated by high $Q$, but only so far, which is why the phone has to lie on the pad.

**Worked example 11 (coupling against distance).** Two rings of radius $R = 2\,$cm made of wire of radius $0.4\,$mm. The cell prints $k$ at separations of 2, 5, 10, 20 and $40\,$mm: from $0.56$ at $2\,$mm it halves roughly every few millimetres, reaching $0.09$ at one coil radius and $0.03$ at two.


In [ ]:
Rc, a_wire = 0.02, 0.4e-3
for dd in (0.002, 0.005, 0.01, 0.02, 0.04):
    print(f"worked example 11, two coils of radius 2 cm, {dd * 1e3:4.0f} mm apart: k = {ind.coaxial_loops_M(Rc, Rc, dd) / ind.ring_self_L(Rc, a_wire):.3f}")
dd = np.linspace(0.001, 0.06, 200)
plt.plot(dd * 1e3, ind.coaxial_loops_M(Rc, Rc, dd) / ind.ring_self_L(Rc, a_wire))
plt.xlabel("separation [mm]"); plt.ylabel("k"); plt.title("coupling of two 2 cm coils"); plt.show()

### 3.4 Energy in magnetic fields

**One inductor.** To raise the current in an inductor from 0 to $I$, the source must push against the back emf. At each instant it delivers power $P = I\,\mathcal{E}_{\text{back}} = IL\,dI/dt$ beyond what the resistance takes, so the work done is
$$W = \int_0^I LI'\,dI' = \tfrac12LI^2.$$
This work is stored, not dissipated. When the source is removed and the circuit is closed through a resistor, the decaying current of the RL circuit (M5 primer 00c) delivers exactly $\tfrac12LI^2$ to the resistor. Compare the capacitor's $\tfrac12CV^2$.

**Two coupled circuits.** Build the currents one at a time. First raise $I_1$ with circuit 2 open: work $\tfrac12L_1I_1^2$. Then raise $I_2$ from 0 to its final value while holding $I_1$ fixed. Source 2 does $\tfrac12L_2I_2^2$ against its own back emf. Meanwhile the changing $I_2$ induces an emf $M\,dI_2/dt$ in circuit 1, and source 1 must work against it to keep $I_1$ constant, delivering $\int I_1M\,dI_2 = MI_1I_2$. In total,
$$U = \tfrac12L_1I_1^2 + MI_1I_2 + \tfrac12L_2I_2^2.$$
Doing it in the other order gives the same expression with $M_{21}$ in place of $M_{12}$. The stored energy cannot depend on the order in which the currents were switched on, which is a second proof of reciprocity.

**Worked example 12 (why $|k| \le 1$).** The energy is a sum of $B^2/2\mu_0$ over all space (below), so $U \ge 0$ for every choice of $I_1$ and $I_2$. Divide by $I_2^2$ and write $x = I_1/I_2$: $\tfrac12L_1x^2 + Mx + \tfrac12L_2 \ge 0$ for all $x$. A quadratic that never goes negative has a discriminant that is not positive: $M^2 - L_1L_2 \le 0$. So $M^2 \le L_1L_2$ and $|k| = |M|/\sqrt{L_1L_2} \le 1$. Equality would need some current ratio with zero field everywhere, which means the two coils' fields cancel exactly, possible only if they share all their flux.

**The energy density.** Apply the one-inductor result to a long solenoid: $U = \tfrac12\dfrac{\mu_0N^2A}{l}I^2$. Rewrite with $B = \mu_0NI/l$:
$$U = \frac{(\mu_0NI/l)^2}{2\mu_0}Al = \frac{B^2}{2\mu_0}\times(\text{volume inside the solenoid}).$$
The energy is exactly what you would get by assigning each cubic metre of field an energy density
$$u_B = \frac{B^2}{2\mu_0},$$
and that statement holds for any static current distribution, not just a solenoid. It is the magnetic twin of $u_E = \tfrac12\epsilon_0E^2$ from M2 primer 00c, and the two together give the energy carried by an electromagnetic wave in primer 00c of this module.

**Worked example 13 (the solenoid both ways).** The solenoid of worked example 1 ($L = 0.987\,$mH, $n = 5000\,$m$^{-1}$) carries $2\,$A. From the inductance: $U = \tfrac12\cdot0.987\times10^{-3}\cdot4 = 1.974\,$mJ. From the field: $B = \mu_0nI = 1.2566\times10^{-6}\cdot5000\cdot2 = 0.01257\,$T, $u_B = B^2/2\mu_0 = 62.83\,\mathrm{J/m^3}$, volume $\pi R^2l = 3.1416\times10^{-5}\,\mathrm{m^3}$, product $1.974\,$mJ.

The energy route also gives inductances that the flux route makes awkward. Inside a solid round wire of radius $a$ carrying a uniformly spread current $I$, Ampère's law gives $B = \mu_0Ir/2\pi a^2$. The energy per unit length inside the wire is $\int_0^a\frac{1}{2\mu_0}\left(\frac{\mu_0Ir}{2\pi a^2}\right)^22\pi r\,dr = \frac{\mu_0I^2}{16\pi}$, and setting this equal to $\tfrac12L_{\text{int}}I^2$ gives $L_{\text{int}} = \mu_0/8\pi$ per unit length, the $0.05\,\mu$H/m added to the coaxial cable in worked example 3.


**The energy of a real, finite solenoid.** The cell integrates $B^2/2\mu_0$ from the Biot-Savart field of an actual helix: 200 turns, radius $2\,$cm, length $40\,$cm, $2\,$A. Two practical points. First, a wire drawn as a line has a field that diverges near the line, so a volume integral reaching the windings would pick up the same logarithmic divergence as the ring in worked example 4. The bore field of a long solenoid is uniform across the cross-section, so the integral is taken over $r < 0.8R$ and scaled by $1/0.8^2$, which stays clear of the wire. Second, the helix is not exactly symmetric about its axis, so the integrand is averaged over four azimuthal angles. The integral runs from $3R$ beyond one end to $3R$ beyond the other.

Three numbers are compared with the result. The ideal $\tfrac12LI^2$ uses $L = \mu_0N^2A/l$. The total stored energy of the real coil is $\tfrac12LI^2$ with the finite-length factor of worked example 1, 0.959 for $2R/l = 0.1$. A quick estimate uses the on-axis field of a finite solenoid (M4 primer 00b) and assumes it is uniform across the bore.


In [ ]:
N, R, l, I0 = 200, 0.02, 0.40, 2.0
mid, dl = bs.segments(bs.helix(R, l, N, 40), closed=False)
f, nr, nz = 0.8, 12, 240
zmax = l / 2 + 3 * R
r = (np.arange(nr) + 0.5) * f * R / nr
z = -zmax + (np.arange(nz) + 0.5) * 2 * zmax / nz
rr, zz = np.meshgrid(r, z, indexing="ij")
dV = 2 * np.pi * rr * (f * R / nr) * (2 * zmax / nz)
U_core = 0.0
for phi in np.linspace(0, 2 * np.pi, 4, endpoint=False):      # four azimuths smooth over the helix pitch
    P = np.stack([rr * np.cos(phi), rr * np.sin(phi), zz], axis=-1)
    U_core += np.sum(np.sum(bs.wire_B(mid, dl, I0, P) ** 2, axis=-1) / (2 * MU0) * dV) / 4
U_inside = U_core / f**2
U_ideal = 0.5 * ind.solenoid_L(N, R, l) * I0**2
zf = np.linspace(-zmax, zmax, 4001)
U_axis = np.trapezoid(bs.solenoid_axis_B(R, l, N / l, I0, zf) ** 2 / (2 * MU0) * np.pi * R**2, zf)
print("worked example 13: U = (1/2) L I^2 = %.3f mJ; u_B x volume = %.3f mJ" % (
    0.5 * ind.solenoid_L(500, 0.01, 0.10) * 4 * 1e3, (MU0 * 5000 * 2) ** 2 / (2 * MU0) * np.pi * 0.01**2 * 0.10 * 1e3))
print(f"\nfinite solenoid, 200 turns, R = 2 cm, l = 40 cm, I = 2 A")
print(f"(1/2) L_ideal I^2                                       = {U_ideal * 1e3:.4f} mJ")
print(f"field energy inside the windings, from wire_B           = {U_inside * 1e3:.4f} mJ   ({U_inside / U_ideal:.4f} of ideal)")
print(f"same, assuming the axis field fills the bore            = {U_axis * 1e3:.4f} mJ   ({U_axis / U_ideal:.4f})")
print(f"total, (1/2) L I^2 with the finite-length factor        = {U_ideal * nagaoka(R, l) * 1e3:.4f} mJ   ({nagaoka(R, l):.4f})")

The field energy inside the windings is 0.936 of the ideal value, and the total energy of the finite coil is 0.959 of it, so about 2 percent of the energy lives outside the bore: in the fringing field beyond the ends and the weak return field around the outside. The axis-only estimate comes out about 1 percent below the full Biot-Savart integral, because near the ends the field acquires a radial component and its axial part varies across the bore, and the axis value misses both.

### 3.5 Magnetic forces from energy

Two circuits carry currents $I_1$ and $I_2$ held constant by their sources. Let one of them move by $dx$ along some direction, so that the mutual inductance changes by $dM$. Three things happen.

1. The field energy changes. With fixed currents only the middle term of $U$ changes: $dU = I_1I_2\,dM$.
2. The flux through each circuit changes, which induces emfs. The flux linkage of circuit 1 changes by $I_2\,dM$, and to keep $I_1$ fixed its source must supply extra work $I_1I_2\,dM$. Circuit 2 needs the same. The sources together supply $2I_1I_2\,dM$.
3. The magnetic force $F$ does mechanical work $F\,dx$.

Energy conservation: what the sources supply goes into field energy plus mechanical work, $2I_1I_2\,dM = I_1I_2\,dM + F\,dx$. So
$$F = I_1I_2\frac{dM}{dx}.$$
The sources supply twice the field-energy change, and half of it goes into motion. The force points towards increasing $M$: circuits move so as to share more flux. For two coaxial loops with currents in the same sense, $M$ grows as they approach, so they attract, which is the attraction of parallel currents (M4 primer 00a) in another form. Reverse one current and they repel. The rule is the same for a coil's own shape: $L$ increases if a solenoid's turns are pushed apart radially, so the windings feel an outward force.

**Worked example 14 (the force between two loops).** Two loops of radius $5\,$cm, $3\,$cm apart on a common axis, each carrying $10\,$A in the same sense. The slope of the exact $M(d)$ by a central difference gives $F = I_1I_2\,dM/dd = -1.617\times10^{-4}\,$N, the minus sign meaning a pull towards each other. The cell compares this with a completely different calculation, the Lorentz force $\sum I_2\,\Delta\mathbf{l}_2\times\mathbf{B}_1$ on loop 2 in the Biot-Savart field of loop 1 (M4 primer 00a), at five separations. The force is small: the weight of $16.5\,$mg.

**Worked example 15 (magnetic pressure).** The same energy argument applied to the windings of a long solenoid gives an outward pressure equal to the energy density, $p = B^2/2\mu_0$. If the radius grows by $dr$ at fixed current, the field energy grows by $u_B\cdot2\pi rl\,dr$ and the force on the area $2\pi rl$ is $u_B$ per unit area. For the solenoid of worked example 13 at $0.0126\,$T: $63\,$Pa, negligible. At $1\,$T: $3.98\times10^5\,$Pa, about 4 atmospheres. In a $3\,$T MRI magnet: $3.58\times10^6\,$Pa, about 35 atmospheres pressing outward on every winding, which is why such magnets need heavy mechanical support.


In [ ]:
a = b = 0.05
I1 = I2 = 10.0
m1, d1 = bs.segments(bs.loop(a, 400))
print("worked example 14, two 5 cm loops, 10 A each")
print("   d [cm]   I1 I2 dM/dd [N]    direct sum of I2 dl x B1 [N]")
ds, F_energy, F_direct = np.array([0.01, 0.02, 0.03, 0.05, 0.10]), [], []
for d in ds:
    h = 1e-6
    F_energy.append(I1 * I2 * (ind.coaxial_loops_M(a, b, d + h) - ind.coaxial_loops_M(a, b, d - h)) / (2 * h))
    m2, d2 = bs.segments(bs.loop(b, 400, centre=(0, 0, d)))
    F_direct.append(np.sum(I2 * np.cross(d2, bs.wire_B(m1, d1, I1, m2)), axis=0)[2])
    print(f"  {d * 100:6.1f}   {F_energy[-1]:+.6e}      {F_direct[-1]:+.6e}")
F_energy, F_direct = np.array(F_energy), np.array(F_direct)
print(f"weight equivalent at 3 cm: {abs(F_energy[2]) / 9.81 * 1e6:.1f} mg")
for Bv in (0.0126, 1.0, 3.0):
    print(f"worked example 15, magnetic pressure at B = {Bv} T: {Bv**2 / (2 * MU0):.4g} Pa")

### Exercise: the coupling coefficient of two rings

Replace the body of `coupling_coefficient` with your own. The check uses two rings of radius $R = 5\,$cm made of wire of radius $a = 0.5\,$mm, with $M$ from `coaxial_loops_M` and $L$ from `ring_self_L`. At every separation $k$ must lie between 0 and 1. For a close pair, $d \ll R$, two rings look locally like two parallel wires and the exact $M$ approaches $\mu_0R(\ln(8R/d) - 2)$, so $k$ should approach $(\ln(8R/d) - 2)/(\ln(8R/a) - 7/4)$. Even when the wires touch, $d = 2a$, this is below 1: the numerator is the denominator minus $\ln2 + \tfrac14$.


In [ ]:
def coupling_coefficient(M, L1, L2):
    '''k = M / sqrt(L1 L2).'''
    return M / np.sqrt(L1 * L2)          # replace with your own version


Rr, aw = 0.05, 0.5e-3
Lr = ind.ring_self_L(Rr, aw)
for d in (1e-3, 2e-3, 5e-3, 20e-3, 100e-3):
    kk = coupling_coefficient(ind.coaxial_loops_M(Rr, Rr, d), Lr, Lr)
    print(f"d = {d * 1e3:5.1f} mm: k = {kk:.4f}")
    assert 0 < kk < 1
d = 1e-3                                 # the wires touch: d = 2a
k_close = coupling_coefficient(ind.coaxial_loops_M(Rr, Rr, d), Lr, Lr)
k_asym = (np.log(8 * Rr / d) - 2) / (np.log(8 * Rr / aw) - 1.75)
print(f"touching rings: k = {k_close:.4f}, close-pair formula {k_asym:.4f}")
assert abs(k_close / k_asym - 1) < 1e-2

## 4. Verify

Two independent checks with asserts. First, the Neumann double sum against the exact coaxial formula for three geometries: equal loops close together, unequal loops at a moderate distance, and a small loop inside a large one. Second, the force from the energy derivative $I_1I_2\,dM/dd$ against the direct Lorentz-force sum at the five separations of worked example 14. The two force calculations share no code: one differentiates an elliptic-integral formula, the other sums cross products of Biot-Savart fields.


In [ ]:
cases = [(0.05, 0.05, 0.005), (0.03, 0.08, 0.06), (0.01, 0.20, 0.0)]
for a, b, d in cases:
    Mn = ind.neumann_M(bs.loop(a, 300), bs.loop(b, 400, centre=(0, 0, d)))
    Me = ind.coaxial_loops_M(a, b, d)
    print(f"a = {a}, b = {b}, d = {d}: Neumann / exact = {Mn / Me:.6f}")
    assert abs(Mn / Me - 1) < 1e-3
rel = np.abs(F_direct / F_energy - 1)
print("force, |direct / energy - 1| at each separation:", ", ".join(f"{x:.1e}" for x in rel))
assert np.all(rel < 1e-4)
print("both checks passed")

## 5. Explore and rebuild

Explore. The Neumann formula fails for a wire with itself, but a standard trick rescues it: compute the mutual inductance between the wire's centre line and a parallel copy displaced by a distance comparable to the wire radius. For a ring of radius $R$, take a coplanar coaxial ring of radius $R - g$ and compute $M(g)$ with `neumann_M`, refining the segments as $g$ shrinks so that each segment stays shorter than $g$. Find the offset $g$ for which $M$ equals `ring_self_L(R, a)`, and express it as a multiple of $a$. Does the multiple depend on $R/a$? Then tilt one of two coaxial loops about a diameter and find the angle at which $M$ passes through zero: a pair of coils that cannot talk to each other, used to isolate the transmit and receive coils of a metal detector.

Rebuild. Close this notebook. On paper, derive $L$ for the solenoid, the toroid and the coaxial cable; derive the Neumann formula from $\Phi = \oint\mathbf{A}\cdot d\mathbf{l}$; derive $\tfrac12LI^2$ and the two-circuit energy, and from it $|k| \le 1$; derive $F = I_1I_2\,dM/dx$ from the energy bookkeeping. In a blank editor, write the Neumann double sum for two polylines and reproduce the table of worked example 7.


## 6. Key results

| result | formula | holds when |
|---|---|---|
| flux linkage | $\Lambda = N\Phi$ | $N$ turns sharing the same flux |
| self-inductance | $L = \Lambda/I$, $\mathcal{E} = -L\,dI/dt$; component law $V = L\,dI/dt$ | rigid circuit |
| solenoid | $L = \mu_0N^2A/l$, times Nagaoka's factor for a finite coil | long coil, uniform winding |
| toroid | $L = \dfrac{\mu_0N^2h}{2\pi}\ln\dfrac ba$ | rectangular core |
| coaxial cable | $L/\ell = \dfrac{\mu_0}{2\pi}\ln\dfrac ba$ (+ $\mu_0/8\pi$ inside a solid inner wire) | |
| ring of wire | $L = \mu_0R(\ln(8R/a) - 7/4)$ | $a \ll R$, uniform current in the wire |
| mutual inductance | $\Lambda_{21} = MI_1$, $\mathcal{E}_2 = -M\,dI_1/dt$ | rigid circuits |
| Neumann formula | $M = \dfrac{\mu_0}{4\pi}\oint\oint\dfrac{d\mathbf{l}_1\cdot d\mathbf{l}_2}{|\mathbf{r}_1 - \mathbf{r}_2|}$ | separate wires |
| reciprocity | $M_{12} = M_{21}$ | always |
| small loop at the centre of a large one | $M \approx \mu_0\pi a^2/2b$ | $a \ll b$ |
| distant coaxial loops | $M \approx \mu_0\pi a^2b^2/2d^3$ | $d \gg a, b$ |
| coaxial loops, exact | $M = \mu_0\sqrt{ab}[(2/k - k)K(k) - (2/k)E(k)]$, $k^2 = 4ab/((a+b)^2 + d^2)$ | |
| coil on a solenoid | $M = \mu_0N_1N_2A/l$ | secondary inside the uniform field |
| coupling coefficient | $k = M/\sqrt{L_1L_2}$, $\lvert k\rvert \le 1$ | |
| ideal transformer | $V_2/V_1 = N_2/N_1$, $I_1/I_2 = N_2/N_1$ | $k = 1$, no losses, changing flux |
| coupled circuits | $V_1 = R_1I_1 + L_1\dot I_1 + M\dot I_2$, $0 = R_2I_2 + L_2\dot I_2 + M\dot I_1$ | lumped circuits |
| energy in an inductor | $U = \tfrac12LI^2$ | |
| two circuits | $U = \tfrac12L_1I_1^2 + MI_1I_2 + \tfrac12L_2I_2^2$ | |
| energy density | $u_B = B^2/2\mu_0$ | static fields, vacuum |
| force between circuits | $F = I_1I_2\,dM/dx$ | constant currents |
| magnetic pressure | $p = B^2/2\mu_0$ on the windings | long solenoid |


## 7. Warm-up answers

1. $L = \mathcal{E}/(dI/dt) = 0.2/50 = 4\,$mH. The induced emf opposes the change: while the current rises, the emf pushes against it (Lenz's law).
2. The field inside is proportional to $N$ (more turns per length carry more current around the core), and each of the $N$ turns links that field. Two factors of $N$.
3. Yes, exactly. Mutual inductance is symmetric, $M_{12} = M_{21}$, because the Neumann formula treats the two wires identically. For the small loop at the centre of a large one, both equal $\mu_0\pi a^2/2b$ to good approximation.
4. A transformer works through a changing shared flux, $V = N\,d\Phi/dt$. A steady current makes a steady flux and induces nothing in the secondary; it only heats the primary's resistance.
5. In the magnetic field, everywhere the field is non-zero, with density $B^2/2\mu_0$. For a long solenoid nearly all of it is inside the bore.
6. The stored energy $\tfrac12L_1I_1^2 + MI_1I_2 + \tfrac12L_2I_2^2$ can never be negative, because it is the integral of $B^2/2\mu_0$. A quadratic form that is never negative has $M^2 \le L_1L_2$, so $|k| \le 1$.
7. They attract. At constant currents the force is $I_1I_2\,dM/dx$, which points towards increasing $M$, and $M$ grows as the loops approach.
